# NSynth + AFTER + Synesis: evaluación de informativeness con linear probing

Notebook de arranque para continuar desde `Setup_environmnet.ipynb`.

Objetivo del notebook:
1. Montar Drive y entrar en el repositorio de `synesis`.
2. Comprobar que el dataset `NSynth` está accesible.
3. Crear/actualizar el wrapper `synesis.datasets.nsynth.Nsynth`.
4. Comprobar que AFTER puede leer audio crudo con la configuración correcta.
5. Extraer embeddings con `AFTER_Timbre_no_adv`.
6. Cargar embeddings `.pt` y entrenar un **linear probe** para una tarea de NSynth.

Idea conceptual:
- AFTER se mantiene congelado.
- Los embeddings se usan como representación fija.
- El linear probe mide *informativeness*: cómo de accesible es un factor de variación, por ejemplo `instrument_family`, desde el embedding.

> Antes de ejecutar: en Colab selecciona GPU en `Runtime > Change runtime type > GPU`.


## 1. Montar Drive y preparar rutas

**Nota sobre dependencias:** este notebook instala los requisitos de Synesis en la sección 2 y, después, fuerza `numpy==1.26.4`, que es la versión que ha evitado el error `No module named 'numpy.rec'` en Colab. No conviene instalar NumPy 2.x en este flujo.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

REPO_ROOT = Path('/content/drive/MyDrive/SYNESIS/synesis')
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/nsynth')

assert REPO_ROOT.exists(), f'No encuentro el repo en {REPO_ROOT}'
assert DATASET_ROOT.exists(), f'No encuentro NSynth en {DATASET_ROOT}'

os.chdir(REPO_ROOT)
print('Repo:', Path.cwd())
print('Dataset:', DATASET_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo: /content/drive/MyDrive/SYNESIS/synesis
Dataset: /content/drive/MyDrive/datasets/nsynth


In [6]:
import sys
from pathlib import Path

print("Directorio actual:", Path.cwd())
print("Existe config?:", (REPO_ROOT / "config").exists())
print("Existe config/features.py?:", (REPO_ROOT / "config" / "features.py").exists())

repo_path = str(REPO_ROOT)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Repo añadido a sys.path:", repo_path)

Directorio actual: /content/drive/MyDrive/SYNESIS/synesis
Existe config?: True
Existe config/features.py?: True
Repo añadido a sys.path: /content/drive/MyDrive/SYNESIS/synesis


## 2. Instalar dependencias e inspeccionar Synesis

In [3]:
# Ejecuta esto si el runtime es nuevo. Puede tardar unos minutos.
# Esta celda instala las dependencias declaradas por Synesis.
!pip install -q -r requirements.txt

# Forzamos esta versión de NumPy porque con versiones 2.x
# nos apareció el error: "No module named 'numpy.rec'".
!pip install -q --force-reinstall "numpy==1.26.4"

# Comprobaciones rápidas de que estamos situados en la raíz correcta del repo.
!pwd
!ls
!find synesis/datasets -maxdepth 1 -type f | sort


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 59.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.8/263.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## 3. Configuración de experimento

Cambia aquí la tarea (`FV`) según el factor de variación de NSynth que quieras predecir.

In [7]:
# Feature extractor que se usará para generar/cargar embeddings.
# AFTER_Timbre será adaptado más abajo para cargar directamente el checkpoint TorchScript (.ts).
FEATURE = 'AFTER_Timbre'

# Factor of Variation (FV): variable que queremos predecir desde el embedding.
# Opciones soportadas por el wrapper:
#   - 'pitch'
#   - 'instrument_family'
#   - 'instrument_source'
#   - 'instrument'
#   - 'qualities'          -> clasificación multi-label
#
# Empezamos con instrument_family. Cambiar FV cambia la tarea del probe,
# no el tipo de embedding extraído.
FV = 'instrument_family'

# Batch pequeño para extracción porque AFTER puede consumir bastante memoria GPU.
BATCH_SIZE_EXTRACT = 16

# Batch para entrenar el probe sobre embeddings ya extraídos.
BATCH_SIZE_TRAIN = 64

# Semilla para mantener resultados más reproducibles.
SEED = 42

print('Feature:', FEATURE)
print('Tarea / factor de variación:', FV)


Feature: AFTER_Timbre
Tarea / factor de variación: instrument_family


## 4. Crear/actualizar el dataset wrapper `Nsynth`

Esta celda escribe el archivo `synesis/datasets/nsynth.py` dentro del repositorio.

Qué hace el wrapper:
- Localiza los `.wav` de NSynth.
- Lee las etiquetas desde `examples.json`.
- Permite cargar audio crudo para extraer embeddings.
- Permite cargar embeddings `.pt` ya extraídos para entrenar probes.
- Mantiene una codificación de etiquetas consistente entre `train`, `validation` y `test`.

Este último punto es importante: si cada split codifica sus clases por separado, la misma clase podría tener números distintos en train y test.

In [8]:
%%writefile /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py
from pathlib import Path
from typing import Optional, Tuple, Union
import json

import torch
from sklearn.preprocessing import LabelEncoder
from torch import Tensor
from torch.utils.data import Dataset

from config.features import configs as feature_configs
from synesis.datasets.dataset_utils import load_track


class Nsynth(Dataset):
    """Wrapper mínimo de NSynth para Synesis.

    Este dataset permite trabajar en dos modos:
    1. item_format='raw': devuelve audio .wav para que Synesis/AFTER pueda extraer embeddings.
    2. item_format='feature': devuelve embeddings .pt ya extraídos para entrenar probes.

    La etiqueta se elige con `fv` (factor de variación). En este TFG nos interesa
    especialmente evaluar si el embedding contiene información relacionada con
    instrumento y características tímbricas.
    """


    def __init__(
        self,
        feature: str,
        root: Union[str, Path] = "/content/drive/MyDrive/datasets/nsynth",
        split: Optional[str] = None,
        download: bool = False,
        feature_config: Optional[dict] = None,
        audio_format: str = "wav",
        item_format: str = "feature",
        itemization: bool = True,
        fv: str = "pitch",
        label: Optional[str] = None,
        seed: int = 42,
    ) -> None:
        # Tareas declaradas para mantener compatibilidad con la estructura de Synesis.
        self.tasks = [
            "pitch_classification",
            "instrument_family_classification",
            "instrument_source_classification",
            "qualities_classification",
        ]

        # Factores de variación de NSynth que este wrapper puede leer desde examples.json.
        self.fvs = [
            "pitch",
            "instrument_family",
            "instrument_source",
            "instrument",
            "qualities",
        ]
        assert fv in self.fvs, (
            f"Invalid factor of variation: {fv}. Options: {self.fvs}"
        )
        self.fv = fv

        # Synesis pasa internamente el argumento `label` al construir datasets.
        # En este wrapper no lo usamos directamente, pero lo aceptamos para mantener
        # compatibilidad con el framework.
        self.label = label

        self.root = Path(root)
        if split not in [None, "train", "test", "validation"]:
            raise ValueError(
                f"Invalid split: {split}. "
                "Options: None, 'train', 'test', 'validation'"
            )

        self.split = split
        self.item_format = item_format
        self.itemization = itemization
        self.audio_format = audio_format
        self.feature = feature
        self.seed = seed

        if download:
            raise NotImplementedError("Download not supported for NSynth")

        # Usamos la configuración del feature extractor, pero permitimos sobrescribirla
        # desde el notebook para fijar explícitamente sample_rate e item_len_sec.
        if not feature_config:
            feature_config = feature_configs[feature]
        self.feature_config = dict(feature_config)

        # Valores necesarios para NSynth.
        # Sus notas duran 4 segundos y en este pipeline usamos 44.1 kHz.
        self.feature_config.setdefault("item_len_sec", 4)
        self.feature_config.setdefault("sample_rate", 44100)

        # LabelEncoder solo es necesario para clasificación single-label.
        # En el caso de 'qualities', NSynth ya proporciona un vector multi-label binario.
        self.label_encoder = LabelEncoder()
        self._fit_label_encoder_on_all_splits()
        self._load_metadata()

    def _read_metadata(self, split_dir: Path) -> dict:
        """Lee examples.json si existe; si no existe, devuelve un diccionario vacío."""
        json_path = split_dir / "examples.json"
        if not json_path.exists():
            return {}

        with open(json_path, "r") as f:
            return json.load(f)

    def _label_from_metadata_item(self, item: dict):
        """Extrae la etiqueta correspondiente al factor de variación seleccionado."""
        if self.fv == "pitch":
            return item["pitch"]

        if self.fv == "instrument_family":
            return item["instrument_family_str"]

        if self.fv == "instrument_source":
            return item["instrument_source_str"]

        if self.fv == "instrument":
            return item["instrument_str"]

        if self.fv == "qualities":
            return item["qualities"]

        raise ValueError(f"Unsupported factor of variation: {self.fv}")

    def _fit_label_encoder_on_all_splits(self) -> None:
        """Crea una codificación de etiquetas consistente para todos los splits.

        Para variables single-label como instrument_family, ajustamos el LabelEncoder
        con las clases de train, validation y test. Así evitamos que una misma clase
        reciba índices distintos en cada split.

        Para qualities no hacemos nada aquí, porque ya viene representado como un
        vector binario multi-label y no debe pasar por LabelEncoder.
        """
        if self.fv == "qualities":
            return

        labels = []

        for split in ["train", "validation", "test"]:
            split_dir = self.root / split
            metadata = self._read_metadata(split_dir) if split_dir.exists() else {}

            for item in metadata.values():
                labels.append(self._label_from_metadata_item(item))

        # Fallback solo para pruebas sin metadata.
        # Para entrenamiento real, conviene conservar examples.json.
        if len(labels) == 0:
            labels = [0]

        self.label_encoder.fit(labels)

    def _candidate_feature_paths(self, wav_path: Path, split_dir: Path) -> list[Path]:
        """Devuelve posibles ubicaciones del embedding .pt.

        Según cómo se ejecute Synesis, los embeddings pueden aparecer en rutas como:
        - split/FEATURE/nombre.pt
        - split/audio/FEATURE/nombre.pt
        - split/audio/FEATURE/nombre.pt relativo al wav

        Comprobamos varias opciones para evitar fallos por diferencias de estructura.
        """
        stem = wav_path.stem
        candidates = [
            split_dir / self.feature / f"{stem}.pt",
            split_dir / "audio" / self.feature / f"{stem}.pt",
            wav_path.parent / self.feature / f"{stem}.pt",
        ]

        # Quita duplicados manteniendo el orden.
        unique = []
        seen = set()

        for candidate in candidates:
            candidate_str = str(candidate)

            if candidate_str not in seen:
                unique.append(candidate)
                seen.add(candidate_str)

        return unique

    def _load_metadata(self) -> Tuple[list, torch.Tensor]:
        """Carga rutas de audio, rutas posibles de embeddings y etiquetas."""
        splits = (
            ["train", "validation", "test"]
            if self.split is None
            else [self.split]
        )

        raw_paths = []
        feature_paths = []
        feature_candidates = []
        raw_labels = []

        for split in splits:
            split_dir = self.root / split

            if not split_dir.exists():
                raise FileNotFoundError(f"No existe el split: {split_dir}")

            # NSynth suele guardar los wav dentro de split/audio,
            # pero permitimos también que estén directamente dentro del split.
            audio_dir = split_dir / "audio"
            if not audio_dir.exists():
                audio_dir = split_dir

            metadata = self._read_metadata(split_dir)

            wav_paths = sorted(audio_dir.glob(f"*.{self.audio_format}"))
            if len(wav_paths) == 0:
                print(
                    f"Aviso: no se encontraron archivos .{self.audio_format} "
                    f"en {audio_dir}"
                )

            for wav_path in wav_paths:
                raw_paths.append(str(wav_path))

                candidates = self._candidate_feature_paths(wav_path, split_dir)
                feature_candidates.append([str(candidate) for candidate in candidates])

                existing = next(
                    (candidate for candidate in candidates if candidate.exists()),
                    None,
                )

                # Si todavía no existe ningún .pt, dejamos como ruta esperada la primera opción.
                # __getitem__ dará un error claro si se intenta cargar antes de extraer embeddings.
                feature_paths.append(
                    str(existing if existing is not None else candidates[0])
                )

                note_id = wav_path.stem

                if metadata and note_id in metadata:
                    raw_labels.append(
                        self._label_from_metadata_item(metadata[note_id])
                    )
                else:
                    raise KeyError(
                        f"No encuentro metadata para el ejemplo '{note_id}' "
                        f"en {split_dir / 'examples.json'}"
                    )

        if self.fv == "qualities":
            labels = torch.tensor(raw_labels, dtype=torch.float32)
        else:
            labels = self.label_encoder.transform(raw_labels)
            labels = torch.tensor(labels, dtype=torch.long)

        self.raw_data_paths = raw_paths
        self.feature_paths = feature_paths
        self.feature_candidates = feature_candidates
        self.labels = labels
        self.paths = (
            self.raw_data_paths
            if self.item_format == "raw"
            else self.feature_paths
        )

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        """Devuelve un ejemplo y su etiqueta.

        - En modo raw: carga audio.
        - En modo feature: carga embedding .pt.
        """
        if self.item_format == "raw":
            path = self.raw_data_paths[idx]
        else:
            candidates = [Path(path) for path in self.feature_candidates[idx]]
            existing = next(
                (candidate for candidate in candidates if candidate.exists()),
                None,
            )

            if existing is None:
                checked_paths = "\n".join(str(path) for path in candidates)

                raise FileNotFoundError(
                    "No encuentro el embedding .pt para este ejemplo.\n"
                    "Primero ejecuta la celda de extracción de embeddings:\n"
                    f"    !python -m synesis.extract -f {self.feature} "
                    "-d Nsynth -b <batch_size>\n\n"
                    "Rutas comprobadas:\n"
                    f"{checked_paths}"
                )

            path = str(existing)

        label = self.labels[idx]

        track = load_track(
            path=path,
            item_format=self.item_format,
            itemization=self.itemization,
            item_len_sec=self.feature_config["item_len_sec"],
            sample_rate=self.feature_config["sample_rate"],
        )

        return track, label

Overwriting /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


## 5. Comprobar configuración de AFTER y dataset en crudo

In [1]:
!pip install -q --force-reinstall "numpy==1.26.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.2

In [9]:
import numpy as np
print(np.__version__)

1.26.4


In [10]:
!find /content/drive/MyDrive/TFG_AFTER/data -name "examples.json"

/content/drive/MyDrive/TFG_AFTER/data/nsynth-test/examples.json
/content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-test/examples.json
/content/drive/MyDrive/TFG_AFTER/data/nsynth_extracted_for_rebuild/nsynth-valid/examples.json


In [11]:
from config.features import configs as feature_configs
from synesis.datasets.nsynth import Nsynth

print('Features disponibles:')
print(list(feature_configs.keys()))

feature_config = {
    **feature_configs[FEATURE],
    # NSynth son notas de 4 s. Mantenerlo explícito evita recortes inesperados.
    'item_len_sec': 4,
    # Mantengo 44.1 kHz para ser consistente con los experimentos previos de AFTER.
    'sample_rate': 44100,
}
print('Feature config usada:', feature_config)

ds_raw = Nsynth(
    feature=FEATURE,
    root=DATASET_ROOT,
    split='train',
    item_format='raw',
    fv=FV,
    feature_config=feature_config,
)

print('Número de ejemplos train:', len(ds_raw))

if FV == 'qualities':
    print('Tarea multi-label. Dimensión del vector qualities:', ds_raw.labels.shape[1])
else:
    print('Clases:', list(ds_raw.label_encoder.classes_))

x, y = ds_raw[0]
print('Tipo x:', type(x))
print('Label y:', y)
try:
    print('Shape x:', x.shape)
except Exception as e:
    print('x no tiene shape directo:', e)


Features disponibles:
['MDuo', 'MelSpec', 'AudioMAE', 'VGGishMTAT', 'MULE_1728', 'MULE_512', 'Music2Latent_8192', 'Music2Latent_64', 'PESTO', 'MERT', 'Wav2Vec2', 'HuBERT', 'CLAP', 'Whisper', 'UniSpeech', 'XVector', 'ResNet18_ImageNet', 'ResNet34_ImageNet', 'ResNet50_ImageNet', 'ResNet101_ImageNet', 'ViT_ImageNet', 'ViT_b_16_ImageNet', 'ViT_l_16_ImageNet', 'ViT_b_32_ImageNet', 'ViT_l_32_ImageNet', 'SimCLR', 'DINO', 'DINOv2', 'DINOv2_small', 'DINOv2_base', 'DINOv2_large', 'ViT_MAE', 'CLIP', 'IJEPA', 'AFTER_Timbre', 'AFTER_Structure', 'AFTER_Combined', 'SSVQVAE_Combined', 'SSVQVAE_Structure', 'SSVQVAE_Timbre', 'TSDSAE_Structure', 'TSDSAE_Timbre', 'TSDSAE_Combined', 'AFTER_Timbre_no_adv', 'AFTER_Structure_no_adv', 'AFTER_Timbre_no_augm', 'AFTER_Structure_no_augm']
Feature config usada: {'__cls__': 'AFTER_Timbre', 'sample_rate': 44100, 'extract_kws': {'autoencoder_path': './externals/AFTER/pretrained/AE_slakh.pt', 'checkpoint_path': './externals/AFTER/pretrained/checkpoint1000000_EMA.pt', '

## 6. Adaptar `AFTER_Timbre` para cargar el checkpoint TorchScript y extraer embeddings

En lugar de reconstruir AFTER desde `config.gin` y checkpoints separados, este notebook usa el mismo checkpoint `.ts` que ya se había utilizado en los experimentos previos de visualización. Después, Synesis podrá llamar al extractor `AFTER_Timbre` con normalidad.

In [12]:
%%writefile synesis/features/after_timbre.py
import torch
from torch import nn


class TSModel(torch.nn.Module):
    """Wrapper para trabajar con el checkpoint TorchScript de AFTER.

    Esta clase mantiene la misma interfaz utilizada en los experimentos previos:
    - ae_encode: audio -> representación latente
    - timbre: representación latente -> embedding tímbrico
    """

    def __init__(self, ts_model, emb_model=None):
        super().__init__()
        self.model = ts_model
        self.emb_model = emb_model

    def ae_encode(self, x):
        """Codifica el audio en el espacio latente del autoencoder."""
        if len(x.shape) > 1:
            x = x.reshape(x.shape[0], 1, -1)
        else:
            x = x.reshape(1, 1, -1)

        if self.emb_model is not None:
            return self.emb_model.encode(x)

        return self.model.emb_model_structure.encode(x)

    def ae_decode(self, z):
        """Reconstruye audio a partir de una representación latente."""
        if self.emb_model is not None:
            return self.emb_model.decode(z).squeeze().cpu()

        return self.model.emb_model_structure.decode(z).squeeze().cpu()

    def timbre(self, z):
        """Extrae el embedding tímbrico a partir de la representación latente."""
        return self.model.encoder.forward_stream(z)

    def structure(self, z):
        """Extrae el embedding estructural a partir de la representación latente."""
        return self.model.encoder_time.forward_stream(z)

    def sample(
        self,
        noise,
        z_structure,
        z_timbre,
        guidance_timbre,
        guidance_structure,
        nb_steps,
    ):
        """Genera audio a partir de ruido y condicionamientos."""
        self.model.set_guidance_timbre(guidance_timbre)
        self.model.set_guidance_structure(guidance_structure)
        self.model.set_nb_steps(nb_steps)

        zout = self.model.sample(
            noise,
            time_cond=z_structure,
            cond=z_timbre,
        )

        return zout


class AFTER_Timbre(nn.Module):
    """Extractor de embeddings tímbricos de AFTER para Synesis.

    Esta versión carga directamente el checkpoint TorchScript (.ts) utilizado
    en los experimentos previos de visualización, en lugar de reconstruir el
    modelo desde archivos .gin y checkpoints separados.
    """

    def __init__(self, feature_extractor=True, extract_kws={}):
        super(AFTER_Timbre, self).__init__()

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Ruta fija al checkpoint TorchScript de AFTER.
        ts_path = "/content/drive/MyDrive/TFG_AFTER/checkpoints/after.audio.instruments.ts"

        if not torch.cuda.is_available():
            print("Aviso: AFTER_Timbre se ejecutará en CPU. La extracción puede ser lenta.")

        # En los experimentos previos se usó el autoencoder integrado dentro del .ts.
        autoencoder_path = None

        # Carga del modelo TorchScript.
        ts_model = torch.jit.load(
            ts_path,
            map_location=self.device,
        )
        ts_model = ts_model.eval().to(self.device)

        # Carga opcional de un autoencoder externo.
        if autoencoder_path is not None:
            emb_model = torch.jit.load(
                autoencoder_path,
                map_location=self.device,
            )
            emb_model = emb_model.eval().to(self.device)
        else:
            emb_model = None

        # Misma lógica que en el notebook previo:
        # model = TSModel(ts_model, emb_model=emb_model)
        self.model = TSModel(
            ts_model,
            emb_model=emb_model,
        )
        self.model = self.model.eval().to(self.device)

    @torch.no_grad()
    def forward(self, x):
        """Extrae el embedding tímbrico a partir del audio de entrada."""
        x = x.to(self.device)

        # Audio -> representación latente.
        z = self.model.ae_encode(x)

        # Representación latente -> embedding tímbrico.
        timbre_emb = self.model.timbre(z)

        return timbre_emb.cpu()


Overwriting synesis/features/after_timbre.py


In [13]:
from pathlib import Path

TS_CHECKPOINT = Path('/content/drive/MyDrive/TFG_AFTER/checkpoints/after.audio.instruments.ts')
assert TS_CHECKPOINT.exists(), f'No encuentro el checkpoint TorchScript en {TS_CHECKPOINT}'
print('Checkpoint TorchScript localizado:', TS_CHECKPOINT)


Checkpoint TorchScript localizado: /content/drive/MyDrive/TFG_AFTER/checkpoints/after.audio.instruments.ts


In [14]:
import torch
torch.cuda.is_available()

True

In [15]:
# Este comando usa el extractor AFTER_Timbre del framework Synesis.
# El extractor ha sido reescrito arriba para cargar internamente el checkpoint .ts.
# Baja BATCH_SIZE_EXTRACT si te quedas sin memoria GPU.
!python -m synesis.extract -f $FEATURE -d Nsynth -b $BATCH_SIZE_EXTRACT --device cuda:0


16784it [1:57:32,  2.38it/s]


In [ ]:
!nvidia-smi

Tue May 19 14:47:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |       0MiB /  15360MiB |     26%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 7. Verificar embeddings generados

In [16]:
from pathlib import Path

print('Buscando embeddings generados...')
for split in ['train', 'validation', 'test']:
    split_dir = DATASET_ROOT / split
    candidate_dirs = [
        split_dir / FEATURE,
        split_dir / 'audio' / FEATURE,
    ]
    print('\nSplit:', split)
    for d in candidate_dirs:
        print(d, 'existe:', d.exists(), 'n_pt:', len(list(d.glob('*.pt'))) if d.exists() else 0)

ds_feat = Nsynth(
    feature=FEATURE,
    root=DATASET_ROOT,
    split='train',
    item_format='feature',
    fv=FV,
    feature_config=feature_config,
)

print('\nEjemplos con rutas de feature:', len(ds_feat))
print('Primer path elegido/esperado:', ds_feat.paths[0])
print('Candidatos para el primer ejemplo:')
for p in ds_feat.feature_candidates[0]:
    print(' -', p, 'existe:', Path(p).exists())

existing_count = sum(any(Path(p).exists() for p in cand) for cand in ds_feat.feature_candidates)
print(f'Embeddings existentes para train: {existing_count}/{len(ds_feat)}')

if existing_count == 0:
    print('\n⚠️ Todavía no hay embeddings .pt. Ejecuta primero la celda de extracción:')
    print(f'!python -m synesis.extract -f {FEATURE} -d nsynth -b {BATCH_SIZE_EXTRACT}')
else:
    x_feat, y_feat = ds_feat[0]
    print('\nTipo embedding:', type(x_feat))
    try:
        print('Shape embedding:', x_feat.shape)
    except Exception as e:
        print('No se pudo imprimir shape:', e)
    print('Label:', y_feat)


Buscando embeddings generados...

Split: train
/content/drive/MyDrive/datasets/nsynth/train/AFTER_Timbre existe: True n_pt: 11741
/content/drive/MyDrive/datasets/nsynth/train/audio/AFTER_Timbre existe: False n_pt: 0

Split: validation
/content/drive/MyDrive/datasets/nsynth/validation/AFTER_Timbre existe: True n_pt: 2516
/content/drive/MyDrive/datasets/nsynth/validation/audio/AFTER_Timbre existe: False n_pt: 0

Split: test
/content/drive/MyDrive/datasets/nsynth/test/AFTER_Timbre existe: True n_pt: 2517
/content/drive/MyDrive/datasets/nsynth/test/audio/AFTER_Timbre existe: False n_pt: 0

Ejemplos con rutas de feature: 11741
Primer path elegido/esperado: /content/drive/MyDrive/datasets/nsynth/train/AFTER_Timbre/bass_electronic_018-022-025.pt
Candidatos para el primer ejemplo:
 - /content/drive/MyDrive/datasets/nsynth/train/AFTER_Timbre/bass_electronic_018-022-025.pt existe: True
 - /content/drive/MyDrive/datasets/nsynth/train/audio/AFTER_Timbre/bass_electronic_018-022-025.pt existe: False

## 8. Utilizamos Synesis para extraer los informatives:
